# 07 · Export the fused dataset — two encodings of the same information

Materializes one flat table per study: **microbiology (T1+T2) ⊕ labs ⊕ vitals ⊕
comorbidity ⊕ `x_cxr`**, in
two parallel variants so downstream models (XGBoost, other frameworks, collaborators)
can consume it without any of this repo's code.

Written in three formats: **`.npy`** (fastest to feed a model), **`.parquet`**
(self-contained — names, dtypes, keys, labels, all in one file; and *smaller*, since the
0/1 block goes out as `int8`), and **`.csv`** (a sample by default — see C5 for why a
full CSV at this width is a poor trade).

**Missing values are left blank (`NaN`), not imputed.** `02` median-imputes labs/vitals so
the neural net in `06` can train (a net cannot take `NaN`); this export is for tree models
(XGBoost reads `NaN` natively), so the fill is undone — a value is present only where it
was actually measured, preserving the real distribution instead of a spike at the median.
The `_mask` columns are kept anyway. `NaN` appears **only** in the EHR value columns; micro,
comorbidity, and `x_cxr` are never `NaN`. Set `BLANK_IMPUTED=False` in C1 to get the filled
version instead.

**Units.** `RAW_UNITS=True` (C1) undoes 02's z-score, so labs/vitals come out in real
units (bpm, C, mmol/L). Trees do not need standardization and raw is readable. Set it False
to match exactly what 06 trained on. `x_cxr` stays standardized regardless -- it is a learned
embedding, not a measurement; use STANDARDIZE_XCXR=False (C0) for its raw CLS.

| block | one-hot (v1) | embedding (v2) |
|---|---|---|
| `t1_org` (organism, fresh culture) | 311 | 16 |
| `t1_spec` (specimen, fresh culture) | 82 | 8 |
| `t2_spec` (specimen, result pending) | 82 | 8 |
| **micro subtotal** | **475** | **32** |
| labs (7 value ⊕ 7 mask — incl. `rbc`) | 14 | 14 |
| vitals (6 value ⊕ 6 mask) | 12 | 12 |
| comorbidity (`diabetes`, `hiv`, `mask`) | 3 | 3 |
| `x_cxr` (frozen RAD-DINO CLS, 7:3 pooled) | 768 | 768 |
| **total** | **1272** | **829** |

The exact widths are printed at run time from `ehr_vocab.pkl`, not hardcoded — if `02`
adds a lab or drops a vital, this table is the thing that goes stale, not the code.

**Why T1+T2 only, no T3/T4** — that is what `06`'s D3 measured: adding T1+T2 to labs
lifts Pneumonia `0.4040 → 0.4196` (the largest single-label gain in the grid) and
Consolidation `0.1796 → 0.1893`, exactly the infection labels where cultures *should*
help. Adding **T3** *undoes* it (Pneumonia back to 0.4101, Lung_Lesion 0.2199 → 0.1919)
— stale 7–90d history is noise, not signal. `D3_12` was the best grid config (0.4068).

**T2 carries no organism, by design** (§A4): tier 2 means "specimen sent, result not
back at the time of the film". An organism id there would *be* the leakage the whole
cohort rebuild removed. So T2 contributes `spec` only.

**The two variants hold identical information.** `nn.EmbeddingBag(mode='sum')` over a
set of ids *is* `multi_hot @ W` — and `02`'s `to_ids` de-duplicates through a `set`, so
the sum has no repeats and the identity is exact. This notebook therefore builds the
multi-hot **once** and produces the embedding as `onehot @ W`, rather than as a second
code path that could silently drift. C4 asserts it against a real `EmbeddingBag`.

Embedding weights come from the trained **`D3_12`** checkpoint (`micro_tiers=(1,2)`) —
an untrained bag is random noise, so the encoding is only meaningful from a fitted
model, and `D3_12` is the config whose tiers match this export exactly.


## C0 · Config

In [ ]:
import os, json, pickle
import numpy as np, pandas as pd
import torch, torch.nn as nn

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = '.'
OUT_DIR   = os.path.join(DATA_DIR, 'cohort_v2')
CACHE_DIR = os.path.join(DATA_DIR, 'cxr_cls_cache')
CKPT_DIR  = os.path.join(DATA_DIR, 'fusion_v2')
DS_DIR    = os.path.join(DATA_DIR, 'dataset_v3'); os.makedirs(DS_DIR, exist_ok=True)

EMB_CKPT  = os.path.join(CKPT_DIR, 'fusion_D3_12_best.pt')   # micro_tiers=(1,2)
STANDARDIZE_XCXR = True     # apply 05's train-only mean/std, same as 06 does
DTYPE = np.float32

print('export ->', DS_DIR)
assert os.path.exists(EMB_CKPT), (f'{EMB_CKPT} not found -> run 06 C11 with RUN_ABLATIONS=True '
                                  '(the D3_12 config) so the trained bags exist')

## C1 · Load the pieces (same joins as `06`)

`x_cxr` is aligned to `study_id` and standardized with **05's train-only** stats, so a
downstream model sees exactly what `06` saw. Labs are already standardized by `02`.

In [ ]:
meta   = json.load(open(os.path.join(OUT_DIR, 'cohort_meta.json')))
LABELS = meta['labels']; assert len(LABELS) == 13

ehr = pd.read_parquet(os.path.join(OUT_DIR, 'cohort_study_level.parquet'))
ehr = ehr.sort_values('study_id').reset_index(drop=True)
N   = len(ehr)

X    = np.load(os.path.join(CACHE_DIR, 'study_x_cxr.npy'))
xidx = pd.read_parquet(os.path.join(CACHE_DIR, 'study_x_cxr_index.parquet'))
row_of = dict(zip(xidx.study_id.to_numpy(), xidx.row.to_numpy()))
assert set(ehr.study_id) <= set(row_of), 'a study has no x_cxr -> re-run 05'
X = X[ehr.study_id.map(row_of).to_numpy()]
if STANDARDIZE_XCXR:
    st = np.load(os.path.join(CACHE_DIR, 'x_cxr_train_stats.npz'))
    X = ((X - st['mean']) / st['std'])
X = X.astype(DTYPE)

vocab = pickle.load(open(os.path.join(OUT_DIR, 'ehr_vocab.pkl'), 'rb'))
ORG_V, SPEC_V = vocab['org_vocab_size'], vocab['spec_vocab_size']
MISSING = vocab['reserved']['MISSING']
LABS = vocab['labs']

# BLANK_IMPUTED: leave every median-imputed continuous cell as NaN instead of the fill.
# 02 median-imputes labs/vitals to feed the neural net (which cannot take NaN) and records
# each fill in `_mask`. That fill puts a spike at the median and distorts the distribution.
# This export is for tree models (XGBoost handles NaN natively), so we UNDO the fill:
# value := NaN wherever mask==0. The `_mask` column stays -- redundant now (mask == notna),
# but explicit, and some tools want the indicator. Micro tokens and comorbidity are NOT
# imputed (MISSING is a real token; como=0 is a real "no such code"), so they are untouched.
BLANK_IMPUTED = True

# RAW_UNITS: labs/vitals in real units (bpm, C, mmol/L), not z-scored. Two sources, best first:
#   1. `{c}_raw` -- the native measured value 02 now emits (NaN where not measured). EXACT:
#      no winsorize, no log/exp round-trip, no imputation. This is what we want.
#   2. fallback: de-standardize `{c}_value` (v*sd+mu, expm1 if log). Exact for the log labs,
#      but WINSOR-CLIPPED at the 0.5/99.5 tails for hgb/rbc/vitals -- so it needs a 02 re-run
#      to get `_raw`. Prints a warning when it falls back.
# x_cxr is a learned embedding, left as-is; use STANDARDIZE_XCXR=False (C0) for its raw CLS.
RAW_UNITS = True
LAB_STATS = vocab.get('lab_stats', {})
HAVE_RAW  = RAW_UNITS and all(f'{c}_raw' in ehr.columns for c in (LABS + vocab.get('vitals', [])))
if RAW_UNITS and not HAVE_RAW:
    print('!! RAW_UNITS: no `_raw` columns in the parquet -> falling back to de-standardization\n'
          '   (exact for log labs, winsor-clipped tails for hgb/rbc/vitals). Re-run 02 for exact raw.')

def _destd(c, v):
    s = LAB_STATS.get(c)
    assert s is not None, f'no stats for {c} in ehr_vocab.pkl -> cannot de-standardize; re-run 02'
    x = v.astype('float64') * s['sd'] + s['mu']
    return (np.expm1(x) if s.get('log') else x)

def _raw_col(c):
    if not RAW_UNITS:            return ehr[f'{c}_value'].to_numpy(DTYPE)   # z-scored, as 06 saw
    if f'{c}_raw' in ehr.columns: return ehr[f'{c}_raw'].to_numpy(DTYPE)    # native units, exact
    return _destd(c, ehr[f'{c}_value'].to_numpy()).astype(DTYPE)           # fallback

def block(cols):
    v = np.stack([_raw_col(c) for c in cols], 1).astype(DTYPE)
    m = np.stack([ehr[f'{c}_mask'].to_numpy(DTYPE) for c in cols], 1)
    if BLANK_IMPUTED:
        v = np.where(m == 1, v, np.nan).astype(DTYPE)     # already NaN if from `_raw`; harmless
    return np.concatenate([v, m], 1), int((m == 0).sum())

LAB, n_lab_imp = block(LABS)                              # already standardized by 02

# vitals (02 C7-bis) + comorbidity (02 C7-ter). vocab.get() tolerates an older pickle.
VITALS = [c for c in vocab.get('vitals', []) if f'{c}_value' in ehr.columns]
VIT, n_vit_imp = block(VITALS) if VITALS else (np.zeros((N, 0), DTYPE), 0)
COMO_COLS = [c for c in (vocab.get('comorbidity', []) + ['como_mask']) if c in ehr.columns]
COMO = (np.stack([ehr[c].to_numpy(DTYPE) for c in COMO_COLS], 1)   # 0/1 indicators, never imputed
        if COMO_COLS else np.zeros((N, 0), DTYPE))
EHRX = np.concatenate([LAB, VIT, COMO], 1)                # the whole non-micro EHR block
print(f'labs {LAB.shape[1]} + vitals {VIT.shape[1]} + como {COMO.shape[1]} = {EHRX.shape[1]} cols')
if BLANK_IMPUTED:
    print(f'blanked {n_lab_imp+n_vit_imp:,} imputed cells -> NaN (labs {n_lab_imp:,}, vitals {n_vit_imp:,}); '
          f'{np.isnan(EHRX).mean():.1%} of the EHR block is now NaN, by design')

if RAW_UNITS:
    # proof the inversion is right: medians must land on physiology, not on z-space (~0)
    print('\nraw-unit sanity (median of measured values; must be physiologic, not ~0):')
    _probe = {'lab_wbc': ('G/L', 4, 12), 'lab_creatinine': ('mg/dL', 0.5, 1.5),
              'vital_temp_c': ('C', 36, 38), 'vital_heart_rate': ('bpm', 60, 100)}
    _all = LABS + VITALS
    for c, (u, lo, hi) in _probe.items():
        if c not in _all: continue
        col = _destd(c, ehr[f'{c}_value'].to_numpy())[ehr[f'{c}_mask'].to_numpy() == 1]
        med = float(np.median(col)); ok = lo <= med <= hi
        print(f'   {c:20s} median {med:7.2f} {u:6s} {"OK" if ok else "!! off -- stats mismatch"}')
        assert ok, f'{c} de-standardized median {med:.2f} outside [{lo},{hi}] -> wrong stats'

print(f'studies {N:,} | x_cxr {X.shape} | labs {LAB.shape} | org vocab {ORG_V} | spec vocab {SPEC_V}')

## C2 · Multi-hot the microbiology (T1 org, T1 spec, T2 spec)

A study can carry several organisms/specimens, so this is **multi**-hot, not one-hot —
the row is the set of ids that study saw. A study with no culture is not all-zeros: it
carries the explicit `MISSING` token (id 2). That distinction matters — all-zeros would
conflate *"never cultured"* with *"cultured, nothing to encode"*, which is precisely the
failure the reserved tokens exist to prevent.

In [ ]:
def multi_hot(col, vsize):
    M = np.zeros((N, vsize), DTYPE)
    for i, v in enumerate(ehr[col].to_numpy()):
        ids = np.asarray(v, np.int64).ravel() if v is not None and np.ndim(v) else np.array([MISSING], np.int64)
        if ids.size == 0: ids = np.array([MISSING], np.int64)
        assert ids.max() < vsize and ids.min() >= 0, (col, ids)
        M[i, ids] = 1.0
    return M

MICRO_BLOCKS = [('t1_org', 't1_org_ids', ORG_V), ('t1_spec', 't1_spec_ids', SPEC_V),
                ('t2_spec', 't2_spec_ids', SPEC_V)]        # T2 has NO org, by design (A4)
OH = {name: multi_hot(col, v) for name, col, v in MICRO_BLOCKS}
MICRO_OH = np.concatenate([OH[n] for n, _, _ in MICRO_BLOCKS], 1)
print('micro one-hot:', MICRO_OH.shape, f'({MICRO_OH.shape[1]} = {ORG_V}+{SPEC_V}+{SPEC_V})')

# how often is each block just MISSING? (sanity: micro is sparse -- see 06's D3 finding)
for n, col, v in MICRO_BLOCKS:
    only_missing = (OH[n].sum(1) == 1) & (OH[n][:, MISSING] == 1)
    print(f'  {n:8s} rows that are MISSING-only: {only_missing.mean():6.2%} | '
          f'mean ids/study {OH[n].sum(1).mean():.2f}')

## C3 · Embedding = `onehot @ W`, weights from the trained `D3_12`

The bag weights live in the checkpoint under `enc.mods.<block>.weight`. Because
`EmbeddingBag(mode='sum')` sums the rows of `W` for a study's ids, and those ids are a
de-duplicated set, `multi_hot @ W` reproduces it **exactly** — asserted below against a
real `EmbeddingBag`, not assumed.

In [ ]:
ck = torch.load(EMB_CKPT, map_location='cpu', weights_only=False)
sd = ck['model']
print('checkpoint:', os.path.basename(EMB_CKPT), '| val mAP at save:', ck.get('val_mAP'))
print('cfg micro_tiers:', ck['cfg'].get('micro_tiers'), '| ehr_cat_encoding:', ck['cfg'].get('ehr_cat_encoding'))
assert tuple(ck['cfg']['micro_tiers']) == (1, 2), 'checkpoint is not the T1+T2 config'
assert ck['cfg']['ehr_cat_encoding'] == 'embedding', 'checkpoint has no EmbeddingBag weights'

W = {}
for name, _, vsize in MICRO_BLOCKS:
    key = f'enc.mods.{name}.weight'
    assert key in sd, f'{key} missing from checkpoint (keys: {[k for k in sd if "mods" in k]})'
    W[name] = sd[key].numpy().astype(DTYPE)
    assert W[name].shape[0] == vsize, (name, W[name].shape, vsize)
    print(f'  {name:8s} W {tuple(W[name].shape)}')

# EXACTNESS CHECK: multi_hot @ W must equal the real EmbeddingBag on the same ids
for name, col, vsize in MICRO_BLOCKS:
    bag = nn.EmbeddingBag(vsize, W[name].shape[1], mode='sum')
    bag.weight.data = torch.from_numpy(W[name])
    probe = np.random.default_rng(0).choice(N, 64, replace=False)
    flat, off, k = [], [], 0
    for i in probe:
        v = ehr[col].to_numpy()[i]
        ids = np.asarray(v, np.int64).ravel() if v is not None and np.ndim(v) else np.array([MISSING], np.int64)
        if ids.size == 0: ids = np.array([MISSING], np.int64)
        flat += ids.tolist(); off.append(k); k += len(ids)
    with torch.no_grad():
        ref = bag(torch.tensor(flat), torch.tensor(off)).numpy()
    got = OH[name][probe] @ W[name]
    assert np.allclose(ref, got, atol=1e-4), (name, np.abs(ref-got).max())
    print(f'  {name:8s} onehot@W == EmbeddingBag  OK  (max diff {np.abs(ref-got).max():.2e})')

MICRO_EM = np.concatenate([OH[n] @ W[n] for n, _, _ in MICRO_BLOCKS], 1)
print('micro embedding:', MICRO_EM.shape)

## C4 · Assemble + write

`.npy` for the matrices (plan §B2: *".npy over 768 parquet columns at this width"*),
plus a parquet index carrying `study_id`, `split`, `setting` and the 13 labels, and a
JSON describing every column block so the export is self-documenting.

In [ ]:
V1 = np.concatenate([MICRO_OH, EHRX, X], 1)        # one-hot   micro | ehr | x_cxr
V2 = np.concatenate([MICRO_EM, EHRX, X], 1)        # embedding  micro | ehr | x_cxr

# NaN is allowed ONLY in the EHR block (blanked imputations, by design). Micro and x_cxr
# must never carry NaN -- one there would be a real bug, so they are asserted separately.
n1o, n1e = MICRO_OH.shape[1], MICRO_EM.shape[1]
for tag, Vv, nm in [('v1', V1, n1o), ('v2', V2, n1e)]:
    micro_blk = Vv[:, :nm]; xcxr_blk = Vv[:, nm + EHRX.shape[1]:]
    assert not np.isnan(micro_blk).any(), f'{tag}: NaN in micro block (should be impossible)'
    assert not np.isnan(xcxr_blk).any(),  f'{tag}: NaN in x_cxr block (should be impossible)'
print('v1 one-hot  :', V1.shape, f'{V1.nbytes/1e9:.2f} GB', f'| NaN in EHR block: {np.isnan(V1).mean():.1%}')
print('v2 embedding:', V2.shape, f'{V2.nbytes/1e9:.2f} GB')

EHR_COLS = ([f'{l}_value' for l in LABS] + [f'{l}_mask' for l in LABS]
            + [f'{v}_value' for v in VITALS] + [f'{v}_mask' for v in VITALS]
            + list(COMO_COLS))
assert len(EHR_COLS) == EHRX.shape[1], (len(EHR_COLS), EHRX.shape[1])

def cols_v1():
    c = []
    for name, _, vsize in MICRO_BLOCKS: c += [f'{name}__{i}' for i in range(vsize)]
    c += EHR_COLS
    c += [f'x_cxr_{i}' for i in range(X.shape[1])]
    return c
def cols_v2():
    c = []
    for name, _, _ in MICRO_BLOCKS: c += [f'{name}__e{i}' for i in range(W[name].shape[1])]
    c += EHR_COLS
    c += [f'x_cxr_{i}' for i in range(X.shape[1])]
    return c
c1, c2 = cols_v1(), cols_v2()
assert len(c1) == V1.shape[1] and len(c2) == V2.shape[1]

np.save(os.path.join(DS_DIR, 'v1_onehot.npy'),    V1)
np.save(os.path.join(DS_DIR, 'v2_embedding.npy'), V2)

idx = ehr[['study_id', 'split', 'setting']].copy()
idx['row'] = np.arange(N)
for l in LABELS: idx[l] = ehr[l].to_numpy('int8')
idx.to_parquet(os.path.join(DS_DIR, 'index.parquet'), index=False)

spec = dict(
    n_studies=int(N), labels=LABELS, split_col='split', row_col='row',
    x_cxr=dict(dim=int(X.shape[1]), source='05 study_x_cxr.npy (RAD-DINO CLS, w_f=0.7/w_l=0.3)',
               standardized=bool(STANDARDIZE_XCXR), stats='x_cxr_train_stats.npz (train-only)'),
    labs=dict(dim=int(LAB.shape[1]), names=[f'{l}_value' for l in LABS]+[f'{l}_mask' for l in LABS],
              note='standardized by 02 on train only; mask 1 = measured; value is NaN where not measured'),
    missing_policy=('blank_nan' if BLANK_IMPUTED else 'median_imputed'),
    ehr_units=('raw' if RAW_UNITS else 'zscore_train'),
    raw_source=('native_raw_column' if HAVE_RAW else ('destandardized' if RAW_UNITS else 'na')),
    micro=dict(tiers=[1, 2], note='T2 carries specimen only -- an organism there would be leakage (A4)',
               blocks={n: dict(onehot_dim=int(v), embed_dim=int(W[n].shape[1])) for n, _, v in MICRO_BLOCKS},
               reserved=vocab['reserved'], embed_source=os.path.basename(EMB_CKPT)),
    variants=dict(v1_onehot=dict(file='v1_onehot.npy', dim=int(V1.shape[1]), columns=c1),
                  v2_embedding=dict(file='v2_embedding.npy', dim=int(V2.shape[1]), columns=c2)),
)
json.dump(spec, open(os.path.join(DS_DIR, 'columns.json'), 'w'), indent=1)
print('wrote index.parquet + columns.json')

## C5 · Also write **parquet** (and optionally CSV)

`.npy` is the fastest thing to feed a model, but it is two files that only mean
something together and it carries no column names. **Parquet is self-contained**:
keys + split + labels + every feature in one file, with names and dtypes, readable from
pandas / R / DuckDB / Spark without this repo.

It is also **smaller**, for a reason worth stating: the one-hot block is 0/1, so it is
written as `int8` rather than `float32` — a 4× cut on 475 columns — and parquet then
run-length-encodes the ~99% zeros to nearly nothing. `x_cxr` is dense random-ish
float32 and does not compress, so it dominates whichever format you pick.

**CSV is supported but not recommended at this width.** 211k rows × ~1268 columns of
text is ~2 GB per variant and minutes to write, then it loses every dtype and rounds
your floats on the way out. `CSV_SAMPLE_N` writes a small readable sample instead —
which is what a CSV is usually actually wanted for. Set it to `None` to force the full
dump anyway.

In [ ]:
import gc
WRITE_PARQUET = True
WRITE_CSV     = True
CSV_SAMPLE_N  = 2000        # None = full dump (~2 GB/variant, slow, lossy). A sample is
                            # usually what CSV is for: eyeballing the schema.

n1, n2 = MICRO_OH.shape[1], MICRO_EM.shape[1]
lab_cols = c1[n1 : n1 + EHRX.shape[1]]
x_cols   = c1[n1 + EHRX.shape[1] :]
assert c1[n1:] == c2[n2:], 'labs/x_cxr column names must match across variants'

keys = ehr[['study_id', 'split', 'setting']].reset_index(drop=True)
for l in LABELS: keys[l] = ehr[l].to_numpy('int8')

def build_df(MICRO, mcols, micro_dtype):
    """keys + labels + features in one frame. one-hot micro -> int8: the values are 0/1,
    float32 would quadruple the block for zero extra information."""
    return pd.concat([keys,
                      pd.DataFrame(MICRO.astype(micro_dtype), columns=mcols),
                      pd.DataFrame(EHRX, columns=lab_cols),
                      pd.DataFrame(X,   columns=x_cols)], axis=1)

for tag, MICRO, mcols, mdt in [('v1_onehot',    MICRO_OH, c1[:n1], 'int8'),
                               ('v2_embedding', MICRO_EM, c2[:n2], DTYPE)]:
    df = build_df(MICRO, mcols, mdt)
    print(f'{tag}: frame {df.shape}  in-RAM {df.memory_usage(deep=True).sum()/1e9:.2f} GB')
    if WRITE_PARQUET:
        p = os.path.join(DS_DIR, f'{tag}.parquet')
        df.to_parquet(p, compression='snappy', index=False)
        print(f'   -> {os.path.basename(p):24s} {os.path.getsize(p)/1e6:8.1f} MB')
    if WRITE_CSV:
        sub = df if CSV_SAMPLE_N is None else df.head(CSV_SAMPLE_N)
        suff = '' if CSV_SAMPLE_N is None else f'_sample{CSV_SAMPLE_N}'
        p = os.path.join(DS_DIR, f'{tag}{suff}.csv')
        sub.to_csv(p, index=False, float_format='%.6g')
        print(f'   -> {os.path.basename(p):24s} {os.path.getsize(p)/1e6:8.1f} MB  ({len(sub):,} rows)')
    del df; gc.collect()
print('\nparquet is self-contained: keys + split + setting + 13 labels + features, names included.')

## C6 · Sanity + how to load

The two variants must agree wherever they can: identical `labs`/`x_cxr` blocks, same row
order, same split. Micro differs only in encoding — and C3 already proved that difference
is an exact linear map, not a loss of information.

In [ ]:
n_mic1 = MICRO_OH.shape[1]; n_mic2 = MICRO_EM.shape[1]
# equal_nan=True: the two variants now share NaNs in the EHR block (same blanked cells),
# and NaN != NaN would make a plain allclose fail on identical data.
assert np.allclose(V1[:, n_mic1:], V2[:, n_mic2:], equal_nan=True), 'ehr/x_cxr blocks differ between variants'
print('OK  ehr+x_cxr identical across variants (NaNs in the same place)')
print(f'OK  row order identical (both indexed by index.parquet row)')

# missingness is now IN the data (NaN), not hidden by a fill -- report it per feature
_ehr_names = EHR_COLS
_ehr_mat   = V1[:, n_mic1:n_mic1+len(_ehr_names)]
print('\nNaN rate per EHR feature (blanked imputations = the true missingness):')
for j, nm in enumerate(_ehr_names):
    r = np.isnan(_ehr_mat[:, j]).mean()
    if r > 0: print(f'   {nm:26s} {r:6.1%} missing')
print('   (mask columns and comorbidity are never NaN -- they are 0/1 indicators)')

print('\nsplit:', dict(idx.split.value_counts()))
print('setting:', dict(idx.setting.value_counts()))
print(f'\nmicro one-hot sparsity: {(MICRO_OH!=0).mean():.4%} non-zero')
print('label prevalence (all splits):')
for l in LABELS: print(f'   {l:28s} {idx[l].mean():.4f}')

print('\nfiles in', DS_DIR, ':')
for f in sorted(os.listdir(DS_DIR)):
    print(f'   {f:22s} {os.path.getsize(os.path.join(DS_DIR,f))/1e6:9.1f} MB')

print(r"""
--- how to load ---

# A) parquet -- one self-contained file, names + dtypes included (pandas/R/DuckDB/Spark)
import pandas as pd, json
spec = json.load(open('columns.json'))
df   = pd.read_parquet('v2_embedding.parquet')
feat = spec['variants']['v2_embedding']['columns']          # feature col names, in order
tr   = df.split == 'train'
Xtr, ytr = df.loc[tr, feat].to_numpy('float32'), df.loc[tr, spec['labels']].to_numpy()

# read only what you need -- parquet is columnar, so this never touches the other columns:
# pd.read_parquet('v2_embedding.parquet', columns=['study_id','split']+spec['labels'])

# B) npy -- fastest to feed a model; matrix + index are two files that pair by `row`
import numpy as np
idx = pd.read_parquet('index.parquet')             # study_id, split, setting, row, 13 labels
X   = np.load('v2_embedding.npy')
tr  = idx.split == 'train'
Xtr, ytr = X[idx.row[tr]], idx.loc[tr, spec['labels']].to_numpy()
""")